# Data Preprocessing Pipeline (Granular V8)
This notebook systematically transforms raw river readings into a robust 3-hour continuous sequence matrix for machine learning.


### 1. Library Imports


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


### 2. Load Data & Unify Stations
Merge any typographical errors in station names so histories remain unbroken.


In [ ]:
df = pd.read_csv('extracted_data_new copy 3.csv')

station_mapping = {'NagalagamStreet': 'Nagalagam Street', 'Kalawellawa(Millakanda)': 'Kalawellawa (Millakanda)'}
df['station'] = df['station'].replace(station_mapping)


### 3. Fix Data Types & Drop Unknowns
Convert timestamps to explicit dates and purge unreadable columns.


In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = df.dropna(subset=['datetime'])

df = df.drop(columns=['trend', 'previous_water_level'], errors='ignore')

numeric_cols = ['water_level', 'rainfall_mm', 'minor_flood_level', 'major_flood_level']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')


### 4. Remove Exact Duplicates


In [ ]:
df = df.sort_values(by=['station', 'datetime'])
df = df.drop_duplicates(subset=['station', 'datetime'], keep='last')


### 5. Native Gap Tracking
We track the biological gaps existing *before* we execute mathematical resampling.


In [ ]:
df['time_diff_hours'] = df.groupby('station')['datetime'].diff().dt.total_seconds() / 3600.0
df['gap_flag_6h'] = (df['time_diff_hours'] > 6).astype(int)
df['gap_flag_24h'] = (df['time_diff_hours'] > 24).astype(int)
df['is_real_reading'] = 1


### 6. Strict Grid Resampling & Leak-Free FFill
This isolates each station and mathematically forces it onto a rigid 3-hour grid, preventing lag features from physically collapsing when reading frequency changes.


In [ ]:
df = df.set_index('datetime')
frames = []

for st, group in df.groupby('station'):
    group = group.sort_index()
    group = group[~group.index.duplicated(keep='last')]
    
    # Safely bucket native readings into the mathematical 3H ticks
    num_resampled = group[['water_level', 'rainfall_mm', 'minor_flood_level', 'major_flood_level', 'time_diff_hours', 'gap_flag_6h', 'gap_flag_24h', 'is_real_reading']].resample('3h').mean()
    cat_resampled = group[['station', 'river_basin', 'rainfall_type', 'status']].resample('3h').first()
    
    resampled = pd.concat([num_resampled, cat_resampled], axis=1)
    
    # Stable Features: Carry forward infinitely
    resampled['station'] = resampled['station'].ffill()
    resampled['river_basin'] = resampled['river_basin'].ffill()
    resampled['minor_flood_level'] = resampled['minor_flood_level'].ffill().bfill()
    resampled['major_flood_level'] = resampled['major_flood_level'].ffill().bfill()
    resampled['time_diff_hours'] = resampled['time_diff_hours'].ffill()
    resampled['gap_flag_6h'] = resampled['gap_flag_6h'].ffill()
    resampled['gap_flag_24h'] = resampled['gap_flag_24h'].ffill()
    
    for col in ['rainfall_type', 'status']:
        resampled[col] = resampled[col].ffill().fillna('Unknown')
    
    # Missing Detection flags explicitly highlighting synthetic slots
    resampled['is_real_reading'] = resampled['is_real_reading'].fillna(0)
    resampled['water_level_was_missing'] = (resampled['is_real_reading'] == 0).astype(int)
    resampled['rainfall_was_missing'] = resampled['rainfall_mm'].isna().astype(int)
    
    # NO LEAKAGE FILL: Flatline historical values safely without observing the future (limit=14 Days)
    resampled['water_level'] = resampled['water_level'].ffill(limit=112)
    resampled['rainfall_mm'] = resampled['rainfall_mm'].fillna(0)
    frames.append(resampled)

df = pd.concat(frames).reset_index()


### 7. Time & Seasonal Extraction


In [ ]:
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['day_of_week'] = df['datetime'].dt.dayofweek
df['quarter'] = df['datetime'].dt.quarter
df['is_monsoon'] = df['month'].isin([5, 6, 7, 10, 11, 12]).astype(int)


### 8. Lags & Momentum Generation


In [ ]:
def create_lags(group):
    # Velocity Feature (Delta)
    group['water_level_delta'] = group['water_level'].diff(1)
    
    for lag in [1, 2, 4, 8, 16, 24]:
        group[f'water_level_lag_{lag}'] = group['water_level'].shift(lag)
    for lag in [1, 2, 4, 8]:
        group[f'rainfall_lag_{lag}'] = group['rainfall_mm'].shift(lag)
        
    return group

df = df.groupby('station', group_keys=False).apply(create_lags)


### 9. Rolling Rainfall Memory
Computing deep rain accumulations covering 24 to 48 hours bounds.


In [ ]:
def create_rolling(group):
    group['water_level_roll_mean_3'] = group['water_level'].rolling(3, min_periods=1).mean()
    group['water_level_roll_max_3'] = group['water_level'].rolling(3, min_periods=1).max()
    
    for w in [3, 6, 8, 12, 16]:
        group[f'rainfall_roll_sum_{w}'] = group['rainfall_mm'].rolling(w, min_periods=1).sum()
        
    return group

df = df.groupby('station', group_keys=False).apply(create_rolling)


### 10. Flood Threshold Features


In [ ]:
df['flood_ratio_minor'] = df['water_level'] / df['minor_flood_level']
df['flood_ratio_major'] = df['water_level'] / df['major_flood_level']
df['above_minor_flag'] = (df['water_level'] > df['minor_flood_level']).astype(int)
df['above_major_flag'] = (df['water_level'] > df['major_flood_level']).astype(int)


### 11. Dual Horizon Target Creation
Explicitly building the 3-Hour and 12-Hour futures.


In [ ]:
df['target_water_level_3h'] = df.groupby('station')['water_level'].shift(-1)
df['target_water_level_12h'] = df.groupby('station')['water_level'].shift(-4)


### 12. Strict Clean & Drop
Any rows intersecting with extreme breaks (like the 2025 blank period) are purged.


In [ ]:
df_clean = df.dropna(subset=[
    'target_water_level_3h', 
    'target_water_level_12h', 
    'water_level_lag_24',
    'water_level_delta'
])
print(f"Final Multi-Horizon training set size: {df_clean.shape}")


### 13. Label Encoding & Save


In [ ]:
from sklearn.preprocessing import LabelEncoder
for col in ['station', 'river_basin', 'rainfall_type', 'status']:
    df_clean[col + '_encoded'] = LabelEncoder().fit_transform(df_clean[col].astype(str))

df_clean.to_csv('processed_training_data.csv', index=False)
print("Saved Granular V8!")
